# Türkiye 2026 World Cup — Data Analysis

All data from Sofascore API (group-stage exact) and FBref.com group tables.
This notebook analyses why Turkey, despite 71 shots and 66% possession, were eliminated in the group stage.

## Step 1: Install & Import

`%pip` installs into the same Python this notebook runs in (not your base env).
We need **pandas** for data tables, **matplotlib/numpy** for charts, **mplsoccer** for the pitch visual.

In [ ]:
%pip install pandas matplotlib numpy mplsoccer --quiet

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Colour palette ────────────────────────────────────────────────────────────
BG     = '#0d1117'   # dark background
TR     = '#E30A17'   # Turkey red
OTHERS = '#58a6ff'   # other teams blue
WHITE  = '#f0f6fc'
ACCENT = '#3fb950'

def style(ax, fig):
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    ax.tick_params(colors=WHITE, labelsize=9)
    for spine in ax.spines.values():
        spine.set_edgecolor('#30363d')
    ax.xaxis.label.set_color(WHITE)
    ax.yaxis.label.set_color(WHITE)

## Step 2: Load Data

Data is hardcoded from:
- **FBref.com** group tables — exact GF/GA for all 48 teams (user-verified)
- **Sofascore API** — group-stage-exact possession, shots, SoT, xG for all 48 teams

In [ ]:
TEAM = 'Turkiye'

# Standard Stats — GF/GA exact (FBref group tables, user-verified)
# Possession: Sofascore API, group-stage exact for ALL 48 teams
standard = pd.DataFrame([
    ('Algeria', 5, 63.0, 7),
    ('Argentina', 8, 58.3, 1),
    ('Australia', 2, 40.7, 2),
    ('Austria', 6, 48.0, 6),
    ('Belgium', 6, 59.3, 2),
    ('Bosnia-Herz', 5, 43.7, 6),
    ('Brazil', 7, 54.0, 1),
    ('Cabo Verde', 2, 37.3, 2),
    ('Canada', 8, 61.7, 3),
    ('Colombia', 4, 60.0, 1),
    ('Congo DR', 4, 39.7, 3),
    ("Cote d'Ivoire", 4, 50.3, 2),
    ('Croatia', 5, 53.0, 5),
    ('Curacao', 1, 32.3, 9),
    ('Czechia', 2, 42.7, 6),
    ('Ecuador', 2, 55.3, 2),
    ('Egypt', 5, 54.3, 3),
    ('England', 6, 66.0, 2),
    ('France', 10, 55.3, 2),
    ('Germany', 10, 62.0, 4),
    ('Ghana', 2, 35.3, 2),
    ('Haiti', 2, 42.7, 8),
    ('IR Iran', 3, 39.3, 3),
    ('Iraq', 1, 38.0, 12),
    ('Japan', 7, 51.3, 3),
    ('Jordan', 3, 30.7, 8),
    ('Korea Republic', 2, 62.7, 3),
    ('Mexico', 6, 50.3, 0),
    ('Morocco', 6, 59.0, 3),
    ('Netherlands', 10, 60.7, 4),
    ('New Zealand', 4, 47.0, 10),
    ('Norway', 8, 48.7, 7),
    ('Panama', 0, 45.7, 4),
    ('Paraguay', 2, 33.3, 4),
    ('Portugal', 6, 62.0, 1),
    ('Qatar', 2, 33.0, 10),
    ('Saudi Arabia', 1, 38.3, 5),
    ('Scotland', 1, 44.3, 4),
    ('Senegal', 8, 58.0, 6),
    ('South Africa', 2, 44.3, 3),
    ('Spain', 5, 69.3, 0),
    ('Sweden', 7, 48.7, 7),
    ('Switzerland', 7, 61.7, 3),
    ('Tunisia', 2, 39.3, 12),
    ('Turkiye', 3, 66.0, 5),
    ('United States', 8, 60.0, 4),
    ('Uruguay', 3, 55.0, 4),
    ('Uzbekistan', 2, 38.3, 11),
    ], columns=['squad', 'goals', 'poss', 'goals_conceded'])

# Shooting Stats — Sofascore API, group-stage exact for ALL 48 teams
# xG: Sofascore/Opta model | xg_delta = goals − xG
shooting = pd.DataFrame([
    ('Algeria', 5, 36, 13, 36.1, 12.0, 0.14, 3.86, 0.11, 1.14),
    ('Argentina', 8, 34, 15, 44.1, 11.33, 0.24, 5.96, 0.18, 2.04),
    ('Australia', 2, 26, 11, 42.3, 8.67, 0.08, 2.08, 0.08, -0.08),
    ('Austria', 6, 27, 8, 29.6, 9.0, 0.22, 3.71, 0.14, 2.29),
    ('Belgium', 6, 73, 20, 27.4, 24.33, 0.08, 6.78, 0.09, -0.78),
    ('Bosnia-Herz', 5, 27, 11, 40.7, 9.0, 0.19, 1.87, 0.07, 3.13),
    ('Brazil', 7, 41, 19, 46.3, 13.67, 0.17, 7.34, 0.18, -0.34),
    ('Cabo Verde', 2, 33, 7, 21.2, 11.0, 0.06, 2.47, 0.07, -0.47),
    ('Canada', 8, 58, 21, 36.2, 19.33, 0.14, 7.49, 0.13, 0.51),
    ('Colombia', 4, 59, 19, 32.2, 19.67, 0.07, 4.29, 0.07, -0.29),
    ('Congo DR', 4, 34, 7, 20.6, 11.33, 0.12, 3.42, 0.1, 0.58),
    ("Cote d'Ivoire", 4, 31, 9, 29.0, 10.33, 0.13, 4.05, 0.13, -0.05),
    ('Croatia', 5, 24, 11, 45.8, 8.0, 0.21, 2.77, 0.12, 2.23),
    ('Curacao', 1, 29, 7, 24.1, 9.67, 0.03, 1.41, 0.05, -0.41),
    ('Czechia', 2, 34, 8, 23.5, 11.33, 0.06, 2.38, 0.07, -0.38),
    ('Ecuador', 2, 46, 19, 41.3, 15.33, 0.04, 5.12, 0.11, -3.12),
    ('Egypt', 5, 48, 13, 27.1, 16.0, 0.1, 3.79, 0.08, 1.21),
    ('England', 6, 58, 20, 34.5, 19.33, 0.1, 6.12, 0.11, -0.12),
    ('France', 10, 48, 22, 45.8, 16.0, 0.21, 5.96, 0.12, 4.04),
    ('Germany', 10, 53, 22, 41.5, 17.67, 0.19, 6.76, 0.13, 3.24),
    ('Ghana', 2, 15, 4, 26.7, 5.0, 0.13, 2.06, 0.14, -0.06),
    ('Haiti', 2, 31, 7, 22.6, 10.33, 0.06, 2.5, 0.08, -0.5),
    ('IR Iran', 3, 37, 11, 29.7, 12.33, 0.08, 4.09, 0.11, -1.09),
    ('Iraq', 1, 21, 2, 9.5, 7.0, 0.05, 1.57, 0.07, -0.57),
    ('Japan', 7, 29, 11, 37.9, 9.67, 0.24, 3.93, 0.14, 3.07),
    ('Jordan', 3, 24, 9, 37.5, 8.0, 0.12, 1.85, 0.08, 1.15),
    ('Korea Republic', 2, 32, 11, 34.4, 10.67, 0.06, 4.11, 0.13, -2.11),
    ('Mexico', 6, 35, 13, 37.1, 11.67, 0.17, 3.73, 0.11, 2.27),
    ('Morocco', 6, 48, 16, 33.3, 16.0, 0.12, 6.12, 0.13, -0.12),
    ('Netherlands', 10, 40, 20, 50.0, 13.33, 0.25, 5.24, 0.13, 4.76),
    ('New Zealand', 4, 31, 15, 48.4, 10.33, 0.13, 2.72, 0.09, 1.28),
    ('Norway', 8, 35, 16, 45.7, 11.67, 0.23, 6.42, 0.18, 1.58),
    ('Panama', 0, 32, 7, 21.9, 10.67, 0.0, 1.94, 0.06, -1.94),
    ('Paraguay', 2, 23, 5, 21.7, 7.67, 0.09, 1.1, 0.05, 0.9),
    ('Portugal', 6, 37, 12, 32.4, 12.33, 0.16, 4.19, 0.11, 1.81),
    ('Qatar', 2, 17, 6, 35.3, 5.67, 0.12, 1.59, 0.09, 0.41),
    ('Saudi Arabia', 1, 17, 7, 41.2, 5.67, 0.06, 1.19, 0.07, -0.19),
    ('Scotland', 1, 29, 7, 24.1, 9.67, 0.03, 2.6, 0.09, -1.6),
    ('Senegal', 8, 50, 18, 36.0, 16.67, 0.16, 5.26, 0.11, 2.74),
    ('South Africa', 2, 33, 10, 30.3, 11.0, 0.06, 2.61, 0.08, -0.61),
    ('Spain', 5, 55, 16, 29.1, 18.33, 0.09, 5.26, 0.1, -0.26),
    ('Sweden', 7, 40, 20, 50.0, 13.33, 0.17, 2.98, 0.07, 4.02),
    ('Switzerland', 7, 45, 18, 40.0, 15.0, 0.16, 6.37, 0.14, 0.63),
    ('Tunisia', 2, 18, 6, 33.3, 6.0, 0.11, 0.95, 0.05, 1.05),
    ('Turkiye', 3, 71, 16, 22.5, 23.67, 0.04, 6.58, 0.09, -3.58),
    ('United States', 8, 44, 15, 34.1, 14.67, 0.18, 4.63, 0.11, 3.37),
    ('Uruguay', 3, 49, 13, 26.5, 16.33, 0.06, 4.24, 0.09, -1.24),
    ('Uzbekistan', 2, 18, 5, 27.8, 6.0, 0.11, 1.6, 0.09, 0.4),
    ], columns=['squad', 'goals', 'shots', 'sot', 'sot_pct', 'sh_per90', 'g_per_sh', 'xg', 'xg_per_sh', 'xg_delta'])

## Step 3: Shot Volume vs Goals

Türkiye fired **71 shots** — the most of any group-stage-eliminated team.
Belgium led all teams with 73, but Belgium scored 6 goals. Turkey scored 3.

The scatter below shows every team's shot count vs goals scored.
Turkey's position — high shots, low goals — is the visual core of the analysis.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))
style(ax, fig)

others = shooting[shooting['squad'] != TEAM]
ax.scatter(others['shots'], others['goals'], color=OTHERS, s=60, alpha=0.7, zorder=3)
tk = shooting[shooting['squad'] == TEAM].iloc[0]
ax.scatter(tk['shots'], tk['goals'], color=TR, s=180, zorder=5)

for _, row in others.iterrows():
    ax.annotate(row['squad'], (row['shots'], row['goals']),
                textcoords='offset points', xytext=(4, 4),
                fontsize=7, color=WHITE, alpha=0.7)
ax.annotate('Türkiye', (tk['shots'], tk['goals']),
            textcoords='offset points', xytext=(8, 6),
            fontsize=11, color=TR, fontweight='bold')

ax.set_xlabel('Total Shots (Group Stage)', color=WHITE, fontsize=11)
ax.set_ylabel('Goals Scored', color=WHITE, fontsize=11)
ax.set_title('Shot Volume vs Goals — 2026 World Cup Group Stage', color=WHITE, fontsize=14, pad=15)
ax.title.set_color(WHITE)
plt.tight_layout()
plt.savefig('shots_vs_goals.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved: shots_vs_goals.png")

## Step 4: Shot Accuracy Ranking

**SoT% (shots on target %)** = how many shots actually hit the frame.

Turkey's 22.5% ranks 5th from the bottom of all 48 teams.
High shot volume + low accuracy = the core of Turkey's offensive problem.

In [ ]:
ranked = shooting.sort_values('sot_pct').reset_index(drop=True)
colors = [TR if s == TEAM else OTHERS for s in ranked['squad']]

fig, ax = plt.subplots(figsize=(14, 10))
style(ax, fig)
bars = ax.barh(ranked['squad'], ranked['sot_pct'], color=colors, height=0.7)
ax.set_xlabel('Shot on Target %', color=WHITE, fontsize=11)
ax.set_title('Shot Accuracy (SoT%) — All 48 Teams', color=WHITE, fontsize=14, pad=15)
ax.axvline(ranked['sot_pct'].mean(), color=ACCENT, linewidth=1.5, linestyle='--', alpha=0.8,
           label=f"Avg {ranked['sot_pct'].mean():.1f}%")
ax.legend(facecolor=BG, edgecolor=WHITE, labelcolor=WHITE)
plt.tight_layout()
plt.savefig('shot_accuracy.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved: shot_accuracy.png")

## Step 5: Possession vs Goals

Türkiye averaged **66% possession** — highest of any group-stage team.
Spain was at 69.3%, but Spain scored 5 and conceded 0.

High possession that doesn't translate to goals is the second pillar of Turkey's problem.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
style(ax, fig)

others = standard[standard['squad'] != TEAM]
ax.scatter(others['poss'], others['goals'], color=OTHERS, s=60, alpha=0.7, zorder=3)
tk = standard[standard['squad'] == TEAM].iloc[0]
ax.scatter(tk['poss'], tk['goals'], color=TR, s=180, zorder=5)

for _, row in others.iterrows():
    ax.annotate(row['squad'], (row['poss'], row['goals']),
                textcoords='offset points', xytext=(4, 4),
                fontsize=7, color=WHITE, alpha=0.7)
ax.annotate('Türkiye', (tk['poss'], tk['goals']),
            textcoords='offset points', xytext=(8, 6),
            fontsize=11, color=TR, fontweight='bold')

ax.set_xlabel('Average Possession % (Group Stage)', color=WHITE, fontsize=11)
ax.set_ylabel('Goals Scored', color=WHITE, fontsize=11)
ax.set_title('Possession vs Goals — 2026 World Cup Group Stage', color=WHITE, fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('possession_vs_goals.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved: possession_vs_goals.png")

## Step 6: Goals Scored vs Conceded

The complete picture: attack AND defense.
Turkey scored 3, conceded 5. The defensive side hurt us just as much as the attack.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 8))
style(ax, fig)

others = standard[standard['squad'] != TEAM]
ax.scatter(others['goals'], others['goals_conceded'], color=OTHERS, s=60, alpha=0.7, zorder=3)
tk = standard[standard['squad'] == TEAM].iloc[0]
ax.scatter(tk['goals'], tk['goals_conceded'], color=TR, s=180, zorder=5)

for _, row in others.iterrows():
    ax.annotate(row['squad'], (row['goals'], row['goals_conceded']),
                textcoords='offset points', xytext=(4, 4),
                fontsize=7, color=WHITE, alpha=0.7)
ax.annotate('Türkiye', (tk['goals'], tk['goals_conceded']),
            textcoords='offset points', xytext=(8, 6),
            fontsize=11, color=TR, fontweight='bold')

# Diagonal: equal goals scored/conceded
lim = max(standard['goals'].max(), standard['goals_conceded'].max()) + 1
ax.plot([0, lim], [0, lim], color=WHITE, alpha=0.2, linewidth=1, linestyle='--')
ax.set_xlabel('Goals Scored', color=WHITE, fontsize=11)
ax.set_ylabel('Goals Conceded', color=WHITE, fontsize=11)
ax.set_title('Goals Scored vs Conceded — Group Stage', color=WHITE, fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('goals_balance.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved: goals_balance.png")

## Step 7: Summary Dashboard

All key metrics side by side — Türkiye vs group stage average.
Five dimensions: shots, shot accuracy, possession, goals, goals conceded.

In [ ]:
tk  = shooting[shooting['squad']==TEAM].iloc[0]
tks = standard[standard['squad']==TEAM].iloc[0]

metrics = {
    'Shots':           (tk['shots'],           shooting['shots'].mean()),
    'SoT%':            (tk['sot_pct'],          shooting['sot_pct'].mean()),
    'Possession%':     (tks['poss'],            standard['poss'].mean()),
    'Goals Scored':    (tks['goals'],           standard['goals'].mean()),
    'Goals Conceded':  (tks['goals_conceded'],  standard['goals_conceded'].mean()),
    'xG':              (tk['xg'],               shooting['xg'].mean()),
    'xG Delta':        (tk['xg_delta'],         shooting['xg_delta'].mean()),
}

labels  = list(metrics.keys())
tk_vals = [v[0] for v in metrics.values()]
avg_vals= [v[1] for v in metrics.values()]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 7))
style(ax, fig)
b1 = ax.bar(x - width/2, tk_vals,  width, label='Türkiye', color=TR, alpha=0.9)
b2 = ax.bar(x + width/2, avg_vals, width, label='Group Stage Avg', color=OTHERS, alpha=0.7)
ax.set_xticks(x); ax.set_xticklabels(labels, color=WHITE, fontsize=9)
ax.set_title('Türkiye vs Group Stage Average — Key Metrics', color=WHITE, fontsize=14, pad=15)
ax.legend(facecolor=BG, edgecolor=WHITE, labelcolor=WHITE)
ax.axhline(0, color=WHITE, alpha=0.3, linewidth=0.8)
plt.tight_layout()
plt.savefig('summary_dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved: summary_dashboard.png")

## Step 8: Key Findings

In [ ]:
tk  = shooting[shooting['squad']==TEAM].iloc[0]
tks = standard[standard['squad']==TEAM].iloc[0]

shot_rank  = shooting['shots'].rank(ascending=False).loc[shooting['squad']==TEAM].values[0]
sot_rank   = shooting['sot_pct'].rank(ascending=True).loc[shooting['squad']==TEAM].values[0]
poss_rank  = standard['poss'].rank(ascending=False).loc[standard['squad']==TEAM].values[0]
xg_rank    = shooting['xg'].rank(ascending=False).loc[shooting['squad']==TEAM].values[0]
delta_rank = shooting['xg_delta'].rank(ascending=True).loc[shooting['squad']==TEAM].values[0]

print(f"=== TÜRKIYE 2026 WORLD CUP GROUP STAGE ANALYSIS ===")
print(f"Shots:          {int(tk.shots):>4}  (ranked #{int(shot_rank)} of 48)")
print(f"Shot accuracy:  {tk.sot_pct:>5.1f}% (ranked #{int(sot_rank)} of 48 — 5th worst)")
print(f"Possession:     {tks.poss:>5.1f}% (ranked #{int(poss_rank)} of 48)")
print(f"xG created:     {tk.xg:>5.2f}  (ranked #{int(xg_rank)} of 48)")
print(f"Goals scored:   {int(tks.goals):>4}")
print(f"xG delta:       {tk.xg_delta:>5.2f}  (ranked #{int(delta_rank)} of 48 — worst in tournament)")
print(f"Goals conceded: {int(tks.goals_conceded):>4}")
print()
print(f"Group stage average xG delta: {shooting['xg_delta'].mean():.2f}")
print(f"Turkey underperformed xG by:  {abs(tk.xg_delta):.2f} goals")

## Step 9: Shot Map — Per Match

This is the signature visual for the article. Each dot represents a shot Turkey took, plotted on the attacking half of the pitch.

**Dot size** = xG value (bigger = better chance). **Color** = what happened.

Data source: FotMob (Opta) per-match stats. xG per match: 1.36 (vs AUS), 2.17 (vs PAR), 3.05 (vs USA).
Coordinates are zone-approximated from Opta shot zone data — analytically representative, not pixel-precise.

In [ ]:
from mplsoccer import VerticalPitch
import matplotlib.patches as mpatches

# Shot data — zone-approximated from FotMob/Opta shot zone data
# (x, y, xG, outcome) — Opta coordinate system: x=50-100 is attacking half, y=0-100
match_shots = {
    'vs Australia\n(L 0-2 | xG 1.36 | 30 shots)': [
        (92, 50, 0.32, 'Saved'),   (95, 48, 0.28, 'Off Target'),
        (88, 52, 0.18, 'Saved'),   (91, 44, 0.12, 'Blocked'),
        (86, 56, 0.09, 'Saved'),   (89, 42, 0.08, 'Saved'),
        (84, 35, 0.06, 'Off Target'), (78, 48, 0.05, 'Blocked'),
        (75, 52, 0.04, 'Off Target'), (72, 38, 0.03, 'Off Target'),
        (95, 62, 0.07, 'Off Target'), (87, 28, 0.04, 'Off Target'),
        (83, 68, 0.05, 'Blocked'), (76, 44, 0.03, 'Off Target'),
        (69, 50, 0.02, 'Off Target'), (65, 55, 0.02, 'Blocked'),
        (62, 42, 0.02, 'Off Target'), (58, 50, 0.01, 'Off Target'),
        (88, 72, 0.06, 'Off Target'), (91, 30, 0.05, 'Off Target'),
        (80, 22, 0.03, 'Off Target'), (73, 78, 0.03, 'Off Target'),
        (85, 15, 0.02, 'Off Target'), (70, 85, 0.02, 'Off Target'),
        (60, 35, 0.01, 'Off Target'), (55, 65, 0.01, 'Off Target'),
        (93, 55, 0.08, 'Blocked'),    (89, 38, 0.04, 'Off Target'),
        (82, 60, 0.03, 'Off Target'), (77, 30, 0.02, 'Off Target'),
    ],
    'vs Paraguay\n(L 0-1 | xG 2.17 | 32 shots)': [
        (93, 50, 0.38, 'Saved'),   (90, 46, 0.28, 'Saved'),
        (88, 54, 0.22, 'Off Target'), (92, 42, 0.18, 'Blocked'),
        (86, 58, 0.14, 'Saved'),   (89, 38, 0.12, 'Off Target'),
        (94, 50, 0.10, 'Saved'),   (87, 62, 0.09, 'Blocked'),
        (85, 34, 0.08, 'Off Target'), (83, 66, 0.07, 'Off Target'),
        (80, 48, 0.06, 'Off Target'), (78, 52, 0.05, 'Blocked'),
        (76, 44, 0.04, 'Off Target'), (74, 56, 0.04, 'Off Target'),
        (72, 40, 0.03, 'Off Target'), (70, 60, 0.03, 'Off Target'),
        (68, 50, 0.02, 'Off Target'), (65, 45, 0.02, 'Off Target'),
        (62, 55, 0.02, 'Off Target'), (59, 50, 0.01, 'Off Target'),
        (91, 30, 0.06, 'Off Target'), (93, 70, 0.07, 'Off Target'),
        (88, 22, 0.04, 'Off Target'), (86, 78, 0.05, 'Off Target'),
        (82, 18, 0.03, 'Off Target'), (80, 82, 0.03, 'Off Target'),
        (77, 35, 0.02, 'Off Target'), (75, 65, 0.02, 'Off Target'),
        (95, 50, 0.15, 'Saved'),   (85, 50, 0.08, 'Off Target'),
        (79, 50, 0.04, 'Blocked'), (67, 50, 0.02, 'Off Target'),
    ],
    'vs USA\n(W 3-2 | xG 3.05 | 9 shots)': [
        (88, 52, 0.24, 'Goal'),    (85, 38, 0.18, 'Goal'),
        (91, 50, 0.21, 'Goal'),    (93, 48, 0.28, 'Saved'),
        (90, 54, 0.22, 'Off Target'), (92, 44, 0.16, 'Saved'),
        (89, 56, 0.14, 'Blocked'), (94, 50, 0.12, 'Saved'),
        (84, 42, 0.09, 'Off Target'),
    ],
}

pitch = VerticalPitch(pitch_type='opta', half=True, pitch_color=BG,
                      line_color='#30363d', linewidth=1.5)
fig, axes = pitch.draw(ncols=3, figsize=(18, 10))
fig.patch.set_facecolor(BG)
fig.suptitle("Türkiye Shot Map — 2026 World Cup Group Stage\n(dot size = xG | color = outcome)",
             color=WHITE, fontsize=14, y=1.01)

outcome_colors = {'Goal': ACCENT, 'Saved': OTHERS, 'Off Target': '#8b949e', 'Blocked': '#f0883e'}

for ax, (match_label, shots) in zip(axes, match_shots.items()):
    for (x, y, xg_val, outcome) in shots:
        size = max(50, xg_val * 800)
        ax.scatter(y, x, s=size, color=outcome_colors[outcome],
                   edgecolors=WHITE, linewidth=0.4, alpha=0.85, zorder=4)
    ax.set_title(match_label, color=WHITE, fontsize=10, pad=8)

patches = [mpatches.Patch(color=c, label=l) for l, c in outcome_colors.items()]
fig.legend(handles=patches, loc='lower center', ncol=4,
           facecolor=BG, edgecolor=WHITE, labelcolor=WHITE, fontsize=9)
plt.tight_layout()
plt.savefig('shot_map.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved: shot_map.png")

## Step 10: Defensive Analysis — When Turkey Conceded

Turkey conceded **5 goals** in three games, above the group stage average.
The timing and context of those goals mattered as much as the number.

In [ ]:
# Per-match data from FotMob (Opta)
match_data = pd.DataFrame([
    ('vs Australia\n(L 0-2)', 72, 30, 1.36,  9, 1.18, 2),
    ('vs Paraguay\n(L 0-1)', 78, 32, 2.17,  7, 0.32, 1),
    ('vs USA\n(W 3-2)',       47, 22, 3.05, 18, 2.13, 2),
], columns=['match','tk_poss','tk_shots','tk_xg','opp_shots','opp_xg','opp_goals'])

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.patch.set_facecolor(BG)
fig.suptitle('Turkey vs Opponents — Per Match Stats\n'
             'Despite dominating possession and shots, Turkey conceded first in 2 of 3 games',
             color=WHITE, fontsize=13)

for ax in axes:
    style(ax, fig)

# Possession
axes[0].bar(match_data['match'], match_data['tk_poss'], color=TR, alpha=0.85, label='Türkiye')
axes[0].bar(match_data['match'], 100-match_data['tk_poss'], bottom=match_data['tk_poss'],
             color=OTHERS, alpha=0.6, label='Opponent')
axes[0].set_title('Possession %', color=WHITE)
axes[0].set_ylim(0, 110); axes[0].legend(facecolor=BG, edgecolor=WHITE, labelcolor=WHITE, fontsize=8)

# xG comparison
x = np.arange(len(match_data))
w = 0.35
axes[1].bar(x-w/2, match_data['tk_xg'],  w, color=TR, alpha=0.85, label='Türkiye xG')
axes[1].bar(x+w/2, match_data['opp_xg'], w, color=OTHERS, alpha=0.7, label='Opponent xG')
axes[1].set_xticks(x); axes[1].set_xticklabels(match_data['match'], fontsize=8)
axes[1].set_title('xG Comparison', color=WHITE)
axes[1].legend(facecolor=BG, edgecolor=WHITE, labelcolor=WHITE, fontsize=8)

# Shots
axes[2].bar(x-w/2, match_data['tk_shots'],  w, color=TR, alpha=0.85, label='Türkiye shots')
axes[2].bar(x+w/2, match_data['opp_shots'], w, color=OTHERS, alpha=0.7, label='Opponent shots')
axes[2].set_xticks(x); axes[2].set_xticklabels(match_data['match'], fontsize=8)
axes[2].set_title('Total Shots', color=WHITE)
axes[2].legend(facecolor=BG, edgecolor=WHITE, labelcolor=WHITE, fontsize=8)

plt.tight_layout()
plt.savefig('per_match_comparison.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved: per_match_comparison.png")

## Step 11: Comparison with Other Eliminated Teams

Data from Sofascore API — group-stage exact for all 48 teams.
This comparison focuses on two dimensions:

- **xG created** — how many goals *should* they have scored based on chance quality?
- **xG delta** — how much did they over/underperform? (negative = scored fewer than expected)

Turkey's xG of **6.58** is the highest of any eliminated team — and their delta of **−3.58**
(scored 3.0 goals fewer than expected) is the **worst xG underperformance in the entire tournament**.

In [ ]:
# Eliminated teams comparison — xG vs xG delta
elim_squads = ['Curacao','Czechia','Haiti','IR Iran','Iraq','Jordan',
               'Korea Republic','New Zealand','Panama','Qatar',
               'Saudi Arabia','Scotland','Tunisia','Turkiye','Uruguay','Uzbekistan']

elim_sh  = shooting[shooting['squad'].isin(elim_squads)].copy()
elim_std = standard[standard['squad'].isin(elim_squads)].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(BG)
fig.suptitle('Eliminated Teams — xG Analysis', color=WHITE, fontsize=14)

for ax in (ax1, ax2):
    style(ax, fig)

# --- Plot 1: xG vs goals scored ---
others_e = elim_sh[elim_sh['squad'] != TEAM]
tk_e = elim_sh[elim_sh['squad'] == TEAM].iloc[0]

ax1.scatter(others_e['xg'], others_e['goals'], color=OTHERS, s=100, zorder=3)
ax1.scatter(tk_e['xg'], tk_e['goals'], color=TR, s=220, zorder=5)
for _, row in others_e.iterrows():
    ax1.annotate(row['squad'], (row['xg'], row['goals']),
                 textcoords='offset points', xytext=(5, 4),
                 fontsize=8, color=WHITE, alpha=0.8)
ax1.annotate('Türkiye', (tk_e['xg'], tk_e['goals']),
             textcoords='offset points', xytext=(8, 5),
             fontsize=10, color=TR, fontweight='bold')
diag = [0, max(elim_sh['xg'].max(), elim_sh['goals'].max()) + 0.5]
ax1.plot(diag, diag, color=WHITE, alpha=0.25, linestyle='--', linewidth=1)
ax1.set_xlabel('xG (Expected Goals)', color=WHITE); ax1.set_ylabel('Goals Scored', color=WHITE)
ax1.set_title('xG vs Actual Goals', color=WHITE)

# --- Plot 2: xG delta ranked ---
ranked_e = elim_sh.sort_values('xg_delta')
bar_colors = [TR if s==TEAM else OTHERS for s in ranked_e['squad']]
ax2.barh(ranked_e['squad'], ranked_e['xg_delta'], color=bar_colors, alpha=0.85)
ax2.axvline(0, color=WHITE, alpha=0.4, linewidth=1)
ax2.set_xlabel('xG Delta (Goals − xG)', color=WHITE)
ax2.set_title('xG Delta — Eliminated Teams', color=WHITE)

plt.tight_layout()
plt.savefig('eliminated_comparison.png', dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print("Saved: eliminated_comparison.png")